# CMZ Civet Behavior Classification — Phase 2
### Task 1 (Image-based Behavior Classification): CNN, Active vs Inactive

**Course:** Digital Image Processing — Chiang Mai University
**Team:** CMZ (Chiang Mai Zoo civet behavior project)

**What this notebook does**

1. Downloads the official labeled dataset (the "วิดิโอชะมดทำai" Drive folder named in the Phase 1 brief):
   `https://drive.google.com/drive/folders/1wHgqAzrEckiyO4wlrSPLZ-hcXGjzKRtd`
   — 3 behavior-sorted clip sets: **Eating (5 clips)**, **Locomotion (10 clips)**, **Resting (7 clips)** — 22 clips total.
2. Extracts frames from every clip and relabels them into two classes:
   - **Active** = Eating + Locomotion clips
   - **Inactive** = Resting clips
3. Fine-tunes a pretrained **MobileNetV2** CNN (transfer learning) to classify a frame as Active / Inactive.
4. Evaluates the model (confusion matrix, per-video active/inactive table, accuracy) and saves everything you need for the submission: trained model, results table, charts.

**Submission checklist this notebook fulfills:**
- [x] Train the model (dataset selected: the official labeled clips, not the raw unlabeled camera footage — see markdown note below)
- [x] Explain the AI/image model selected (see the write-up near the end)
- [x] Design and evaluate the output (active/inactive table + confusion matrix + accuracy)
- [ ] Push this notebook + results to GitHub (see the last section for a ready-to-use repo structure)

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a **T4 GPU** (free tier). The whole notebook (download + frame extraction + training) should take well under 15 minutes on GPU.


## Why this dataset, not the raw camera footage

Two datasets were shared for this project:

1. **The official one** (named directly in the Phase 1 brief's "Dataset:" line): short, pre-trimmed clips already sorted into `Eating/`, `Locomotion/`, `Resting/` folders — real ground-truth labels, no guessing needed.
2. **Raw, unlabeled, full-day footage** from cameras 1–4 (several hundred MB per file, no behavior labels at all).

This notebook uses dataset (1). It's the one the course actually points to, it already has trustworthy labels, and it's small enough to train on in minutes instead of hours.


In [ ]:
# 1. Install dependencies (Colab already has TensorFlow; this just adds the extras we need)
!pip install -q gdown opencv-python-headless scikit-learn seaborn


In [ ]:
import os
import glob
import shutil
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Download the official labeled dataset

This pulls the whole `วิดิโอชะมดทำai` folder (Eating / Locomotion / Resting subfolders) straight from Drive.


In [ ]:
FOLDER_ID = "1wHgqAzrEckiyO4wlrSPLZ-hcXGjzKRtd"
RAW_DIR = "cmz_raw"

if os.path.exists(RAW_DIR):
    shutil.rmtree(RAW_DIR)

!gdown --folder "https://drive.google.com/drive/folders/{FOLDER_ID}" -O {RAW_DIR} --quiet

# Sanity check: list what we got
for root, dirs, files in os.walk(RAW_DIR):
    level = root.replace(RAW_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or RAW_DIR}/  ({len(files)} files)")


**If `gdown --folder` fails** (Google occasionally rate-limits folder downloads): download the three subfolders manually from the Drive link as zip files, upload them to this Colab session (folder icon on the left → upload), and unzip them into `cmz_raw/Eating`, `cmz_raw/Locomotion`, `cmz_raw/Resting` — then skip straight to the next cell.


## 3. Extract frames and relabel into Active / Inactive

We sample evenly-spaced frames from every clip (this both turns 22 videos into a few hundred training images, and is a simple form of data augmentation by itself — slightly different frames of the same behavior).


In [ ]:
ACTIVE_FOLDERS = ["Eating", "Locomotion"]
INACTIVE_FOLDERS = ["Resting"]
FRAMES_PER_CLIP = 25

FRAMES_DIR = "cmz_frames"
if os.path.exists(FRAMES_DIR):
    shutil.rmtree(FRAMES_DIR)
os.makedirs(os.path.join(FRAMES_DIR, "active"), exist_ok=True)
os.makedirs(os.path.join(FRAMES_DIR, "inactive"), exist_ok=True)


def extract_frames(video_path, out_dir, tag, n_frames=FRAMES_PER_CLIP):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return 0
    idxs = np.linspace(0, total - 1, min(n_frames, total)).astype(int)
    saved = 0
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if not ok:
            continue
        out_path = os.path.join(out_dir, f"{tag}_f{int(i):05d}.jpg")
        cv2.imwrite(out_path, frame)
        saved += 1
    cap.release()
    return saved


def process_class(folder_names, label):
    out_dir = os.path.join(FRAMES_DIR, label)
    video_log = []
    for folder in folder_names:
        folder_path = os.path.join(RAW_DIR, folder)
        videos = sorted(glob.glob(os.path.join(folder_path, "*.mp4")) +
                         glob.glob(os.path.join(folder_path, "*.MP4")))
        for v in videos:
            # prefix with the source folder so same-named clips from different
            # folders (e.g. two files both called clip1.mp4) never collide
            tag = f"{folder}_{os.path.splitext(os.path.basename(v))[0]}"
            n = extract_frames(v, out_dir, tag)
            video_log.append({"source_folder": folder, "video": os.path.basename(v), "frames_extracted": n})
    return video_log

log_active = process_class(ACTIVE_FOLDERS, "active")
log_inactive = process_class(INACTIVE_FOLDERS, "inactive")

video_log_df = pd.DataFrame(log_active + log_inactive)
print(video_log_df)

n_active = len(glob.glob(os.path.join(FRAMES_DIR, "active", "*.jpg")))
n_inactive = len(glob.glob(os.path.join(FRAMES_DIR, "inactive", "*.jpg")))
print(f"\nTotal frames -> active: {n_active}, inactive: {n_inactive}")


## 4. Build train / validation datasets

**Important:** we split at the *clip* level, not the frame level. If we split frames
randomly, frames from the same clip (which look nearly identical to their neighbors)
could end up in both train and validation, which would inflate validation accuracy
without the model actually generalizing to new clips. So every frame from a given
video goes entirely into either train or validation, never both.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE
class_names = ["active", "inactive"]  # label 0 = active, label 1 = inactive
label_to_idx = {name: i for i, name in enumerate(class_names)}

# Build a manifest of every extracted frame, tagged with the clip it came from
# (the tag is the "<folder>_<videoname>" prefix we used when saving each frame).
records = []
for label in class_names:
    for fp in sorted(glob.glob(os.path.join(FRAMES_DIR, label, "*.jpg"))):
        video_tag = os.path.basename(fp).rsplit("_f", 1)[0]
        records.append({"path": fp, "label": label, "video_tag": video_tag})
frames_df = pd.DataFrame(records)

# Split CLIPS (not frames) 80/20 into train/val, per class, with a fixed seed
rng = np.random.RandomState(SEED)
train_tags, val_tags = [], []
for label in class_names:
    tags = sorted(frames_df.loc[frames_df.label == label, "video_tag"].unique().tolist())
    rng.shuffle(tags)
    n_val = max(1, round(len(tags) * 0.2))
    val_tags.extend(tags[:n_val])
    train_tags.extend(tags[n_val:])

train_df = frames_df[frames_df.video_tag.isin(train_tags)].reset_index(drop=True)
val_df = frames_df[frames_df.video_tag.isin(val_tags)].reset_index(drop=True)

print(f"Train: {len(train_df)} frames from {len(train_tags)} clips -> {sorted(train_tags)}")
print(f"Val:   {len(val_df)} frames from {len(val_tags)} clips -> {sorted(val_tags)}")


def make_dataset(df, shuffle):
    paths = df["path"].values
    labels = df["label"].map(label_to_idx).values.astype("float32").reshape(-1, 1)

    def load_img(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(max(len(df), 1), seed=SEED)
    return ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)  # order preserved, matches val_df row order


In [ ]:
# Quick visual sanity check of a batch
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.clip(images[i].numpy(), 0, 255).astype("uint8"))
        plt.title(class_names[int(labels[i].numpy()[0])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Model: MobileNetV2 transfer learning

Data augmentation + preprocessing are built into the model itself, so the same model works correctly on raw images at inference time with no separate preprocessing script.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # freeze for initial training

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs, name="cmz_behavior_classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 6. Train (frozen base), then fine-tune


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True
)

EPOCHS_HEAD = 15
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[early_stop],
)


In [ ]:
# Fine-tune: unfreeze the top layers of the base model and train a little more with a low LR
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30  # unfreeze roughly the last 30 layers
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_FINE_TUNE = 8
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=[early_stop],
)


In [ ]:
# Combined training curves
acc = history.history["accuracy"] + history_fine.history["accuracy"]
val_acc = history.history["val_accuracy"] + history_fine.history["val_accuracy"]
loss = history.history["loss"] + history_fine.history["loss"]
val_loss = history.history["val_loss"] + history_fine.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(acc, label="train")
axes[0].plot(val_acc, label="val")
axes[0].axvline(len(history.history["accuracy"]) - 1, color="gray", linestyle="--", label="fine-tune start")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(loss, label="train")
axes[1].plot(val_loss, label="val")
axes[1].axvline(len(history.history["loss"]) - 1, color="gray", linestyle="--", label="fine-tune start")
axes[1].set_title("Loss")
axes[1].legend()
plt.tight_layout()
plt.show()


## 7. Evaluate: confusion matrix, classification report, active/inactive table


In [ ]:
# val_ds was built with shuffle=False, so batches come out in the same order
# as val_df's rows - we can match predictions back to filenames directly.
y_true = val_df["label"].map(label_to_idx).values.astype(int)
y_prob = model.predict(val_ds, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows=true, cols=predicted), classes:", class_names)
print(cm)
print()
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — Active vs Inactive")
plt.tight_layout()
plt.show()


In [ ]:
# Per-frame active/inactive results table (the "design and evaluate the output" deliverable)
results_df = pd.DataFrame({
    "file": [os.path.basename(p) for p in val_df["path"]],
    "video_clip": val_df["video_tag"].values,
    "true_label": [class_names[i] for i in y_true],
    "predicted_label": [class_names[i] for i in y_pred],
    "confidence": [p if pred == 1 else 1 - p for p, pred in zip(y_prob, y_pred)],
})
results_df["correct"] = results_df["true_label"] == results_df["predicted_label"]

results_df.to_csv("results_table.csv", index=False)
print(f"Frame-level validation accuracy: {results_df['correct'].mean():.2%}")
results_df.head(15)


In [ ]:
# Per-CLIP active/inactive table (majority vote across that clip's frames) -
# this is the cleaner table to put on a presentation slide: one row per video.
video_level = (
    results_df.groupby(["video_clip", "true_label"])
    .agg(
        predicted_label=("predicted_label", lambda s: s.value_counts().idxmax()),
        mean_confidence=("confidence", "mean"),
        frames_evaluated=("predicted_label", "count"),
    )
    .reset_index()
)
video_level["correct"] = video_level["true_label"] == video_level["predicted_label"]
video_level.to_csv("video_level_table.csv", index=False)
print(f"Clip-level validation accuracy: {video_level['correct'].mean():.2%} "
      f"({video_level['correct'].sum()}/{len(video_level)} held-out clips)")
video_level


In [ ]:
# Summary bar chart: predicted active vs inactive counts
summary = results_df["predicted_label"].value_counts().reindex(class_names, fill_value=0)
plt.figure(figsize=(5, 4))
sns.barplot(x=summary.index, y=summary.values, palette=["#4C9F70", "#C96567"])
plt.ylabel("Number of validation frames")
plt.title("Predicted Active vs Inactive (validation set)")
plt.tight_layout()
plt.show()

summary


## 8. Save the trained model


In [ ]:
model.save("cmz_behavior_classifier.keras")
print("Saved model to cmz_behavior_classifier.keras")

# Optional: save to your own Google Drive instead of just the Colab session
# from google.colab import drive
# drive.mount('/content/drive')
# model.save('/content/drive/MyDrive/cmz_behavior_classifier.keras')
# results_df.to_csv('/content/drive/MyDrive/results_table.csv', index=False)


## 9. The AI/image model we selected — explanation

**Task:** Task 1 from the course brief — Image-based Behavior Classification (`Image Dataset → Labeling → CNN → Behavior Classification`), simplified from the original 4 behavior labels to a binary **Active vs Inactive** classification, matching what was asked for the Phase 2 submission.

**Model: MobileNetV2, used via transfer learning**

- **Why a CNN at all:** convolutional layers learn spatial features (edges, textures, body posture/shape) directly from pixels, which is exactly what distinguishes an eating/moving civet from a resting one — no hand-designed feature extraction needed, unlike the classical Chapter 2–7 pipeline (gamma correction, thresholding, morphology) used in Phase 1.
- **Why MobileNetV2 specifically:** it uses depthwise-separable convolutions, which give it far fewer parameters and FLOPs than a standard CNN of similar depth (e.g. VGG or a vanilla ResNet). That means it trains fast on a free Colab GPU (minutes, not hours) and does not overfit as quickly on a small dataset — both important given our dataset is only 22 source clips.
- **Why transfer learning instead of training from scratch:** MobileNetV2 is pretrained on ImageNet (1.4M images, 1000 classes). Its early/middle layers already encode general-purpose visual features (edges, textures, shapes) that transfer well to a new, unrelated image domain. We freeze those layers, train only a small classification head on our data first, then unfreeze the last ~30 layers for a brief fine-tuning pass at a low learning rate. Training a full CNN from scratch on ~400–500 frames (after frame extraction) would badly overfit.
- **Why frame sampling from video rather than video-level models:** the brief's Task 1 pipeline is explicitly image-based (`Image Dataset → Labeling → CNN`), not video-based (that's Task 2/YOLO or Task 3/pose territory). Sampling evenly-spaced frames turns each clip into many labeled training images while keeping the model and pipeline simple enough to finish and explain clearly under the deadline.
- **Why binary Active/Inactive instead of 4-way (Walking/Eating/Resting/Climbing):** the real labeled dataset only provides 3 categories (Eating, Locomotion, Resting — no separate Climbing clips exist), and the Phase 2 submission instructions explicitly ask for an "active and inactive table or visualization." Mapping Eating + Locomotion → Active and Resting → Inactive uses the real labels as-is, with no guessing, and directly produces the deliverable that's asked for.


## 11. Bonus — Per-Behavior Breakdown (Eating / Locomotion / Resting)

The Phase 2 submission only requires an **active/inactive** table, and the model above fully satisfies that already. This section goes further: it reuses the exact same extracted frames and the exact same clip-level train/validation split from Section 4 (`train_tags` / `val_tags`), so the comparison is apples-to-apples — but trains a second classifier to predict the *specific* behavior (Eating, Locomotion, or Resting) instead of just Active/Inactive.

This is bonus scope beyond the minimum requirement, not a replacement for the active/inactive model above — it's here to show more depth and to feed a richer presentation dashboard.

**Heads-up on expectations:** Eating only has 5 source clips total (about 4 after the 80/20 split), so this 3-class model has much less data per class than the binary model and will likely score lower, especially on Eating. That's a data-size reality, not a bug — worth saying out loud if asked.

**You don't need to re-run Sections 1-3 to get here** if they've already run in this session (frames are cached on disk in `cmz_frames/`) — just make sure Section 4 has run too, since this reuses `frames_df`, `train_tags`, and `val_tags` from it.


In [ ]:
# Every frame's tag is "<SourceFolder>_<videoname>", e.g. "Eating_clip1" - the source
# folder name IS the specific behavior label, so no new extraction or labeling is
# needed, we just read it back out of the tag already assigned in Section 3.
BEHAVIOR_CLASSES = ["Eating", "Locomotion", "Resting"]
behavior_to_idx = {name: i for i, name in enumerate(BEHAVIOR_CLASSES)}

frames_df["behavior"] = frames_df["video_tag"].str.split("_", n=1).str[0]

assert set(frames_df["behavior"].unique()) == set(BEHAVIOR_CLASSES), \
    f"Unexpected behavior labels found: {sorted(frames_df['behavior'].unique())}"

print("Clips per behavior:")
print(frames_df.groupby("behavior")["video_tag"].nunique())
print("\nFrames per behavior:")
print(frames_df["behavior"].value_counts())

# Same clips as the binary model's split (train_tags / val_tags from Section 4) -
# just re-tagged with their specific behavior instead of active/inactive.
train_df3 = frames_df[frames_df.video_tag.isin(train_tags)].reset_index(drop=True)
val_df3 = frames_df[frames_df.video_tag.isin(val_tags)].reset_index(drop=True)

print(f"\nTrain: {len(train_df3)} frames from {len(train_tags)} clips")
print(f"Val:   {len(val_df3)} frames from {len(val_tags)} clips -> {sorted(val_tags)}")


In [ ]:
def make_dataset_3class(df, shuffle):
    paths = df["path"].values
    labels = df["behavior"].map(behavior_to_idx).values.astype("int32")

    def load_img(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(max(len(df), 1), seed=SEED)
    return ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

train_ds3 = make_dataset_3class(train_df3, shuffle=True)
val_ds3 = make_dataset_3class(val_df3, shuffle=False)  # order preserved


### Model: a second MobileNetV2, same recipe, 3-way softmax head

Separate model instance from the binary classifier above (different variable names, `_3` suffix throughout) so neither model's weights interfere with the other's.


In [ ]:
base_model_3 = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model_3.trainable = False

inputs3 = tf.keras.Input(shape=IMG_SIZE + (3,))
x3 = data_augmentation(inputs3)
x3 = tf.keras.applications.mobilenet_v2.preprocess_input(x3)
x3 = base_model_3(x3, training=False)
x3 = layers.GlobalAveragePooling2D()(x3)
x3 = layers.Dropout(0.3)(x3)
outputs3 = layers.Dense(len(BEHAVIOR_CLASSES), activation="softmax")(x3)

model_3class = tf.keras.Model(inputs3, outputs3, name="cmz_behavior_classifier_3class")

model_3class.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model_3class.summary()


In [ ]:
early_stop_3 = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True
)

history_3 = model_3class.fit(
    train_ds3, validation_data=val_ds3, epochs=15, callbacks=[early_stop_3],
)


In [ ]:
# Fine-tune, same recipe as the binary model
base_model_3.trainable = True
FINE_TUNE_AT_3 = len(base_model_3.layers) - 30
for layer in base_model_3.layers[:FINE_TUNE_AT_3]:
    layer.trainable = False

model_3class.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_3_fine = model_3class.fit(
    train_ds3, validation_data=val_ds3, epochs=8, callbacks=[early_stop_3],
)


### Evaluate the 3-class model


In [ ]:
y_true3 = val_df3["behavior"].map(behavior_to_idx).values.astype(int)
y_prob3 = model_3class.predict(val_ds3, verbose=0)
y_pred3 = y_prob3.argmax(axis=1)

cm3 = confusion_matrix(y_true3, y_pred3, labels=list(range(len(BEHAVIOR_CLASSES))))
print("Confusion matrix (rows=true, cols=predicted), classes:", BEHAVIOR_CLASSES)
print(cm3)
print()
print(classification_report(
    y_true3, y_pred3, target_names=BEHAVIOR_CLASSES,
    labels=list(range(len(BEHAVIOR_CLASSES))), zero_division=0,
))

plt.figure(figsize=(5, 4))
sns.heatmap(cm3, annot=True, fmt="d", cmap="Oranges",
            xticklabels=BEHAVIOR_CLASSES, yticklabels=BEHAVIOR_CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — Eating / Locomotion / Resting")
plt.tight_layout()
plt.show()


In [ ]:
results_df3 = pd.DataFrame({
    "file": [os.path.basename(p) for p in val_df3["path"]],
    "video_clip": val_df3["video_tag"].values,
    "true_behavior": [BEHAVIOR_CLASSES[i] for i in y_true3],
    "predicted_behavior": [BEHAVIOR_CLASSES[i] for i in y_pred3],
    "confidence": y_prob3.max(axis=1),
})
results_df3["correct"] = results_df3["true_behavior"] == results_df3["predicted_behavior"]
results_df3.to_csv("results_table_3class.csv", index=False)
print(f"Frame-level 3-class validation accuracy: {results_df3['correct'].mean():.2%}")
results_df3.head(15)


In [ ]:
video_level_3 = (
    results_df3.groupby(["video_clip", "true_behavior"])
    .agg(
        predicted_behavior=("predicted_behavior", lambda s: s.value_counts().idxmax()),
        mean_confidence=("confidence", "mean"),
        frames_evaluated=("predicted_behavior", "count"),
    )
    .reset_index()
)
video_level_3["correct"] = video_level_3["true_behavior"] == video_level_3["predicted_behavior"]
video_level_3.to_csv("video_level_table_3class.csv", index=False)
print(f"Clip-level 3-class accuracy: {video_level_3['correct'].mean():.2%} "
      f"({video_level_3['correct'].sum()}/{len(video_level_3)} held-out clips)")
video_level_3


In [ ]:
model_3class.save("cmz_behavior_classifier_3class.keras")

# Combined table: one row per held-out clip, both models' predictions side by side -
# this is the file to feed a presentation dashboard.
combined = video_level.merge(
    video_level_3[["video_clip", "true_behavior", "predicted_behavior", "mean_confidence"]]
        .rename(columns={"mean_confidence": "mean_confidence_3class",
                          "predicted_behavior": "predicted_behavior_3class"}),
    on="video_clip", how="outer",
)
combined.to_csv("video_level_combined.csv", index=False)
print("Saved cmz_behavior_classifier_3class.keras, results_table_3class.csv,")
print("video_level_table_3class.csv, and video_level_combined.csv")
combined


## 12. Annotated videos — behavior overlaid on the real footage

The Ajarn asked to see the results directly on the actual video, not just in a table: a label in the corner of the frame showing what the model thinks the civet is doing as the video plays, like watching a live detector — e.g. "Eating" shows up while it's eating.

This section runs **both** trained models (`model` for Active/Inactive, `model_3class` for the specific behavior) on **every frame** of the same four held-out validation clips shown in the results tables above, and burns the detected behavior onto the video itself as a text label. Nothing above is changed or re-run — this only adds new output files (videos + a per-frame CSV log).

If you're running this right after Sections 1–11 above in the same Colab session, the original video files are already sitting in `cmz_raw/` from Section 2 — nothing needs to be re-downloaded. If you're resuming in a **fresh** Colab session instead, run the next cell first to reload the saved models and the dataset; otherwise skip it.


In [ ]:
# Only needed in a FRESH Colab session (Sections 1-11 above haven't just run).
# Skip this cell if they have — model, model_3class, RAW_DIR, IMG_SIZE,
# BEHAVIOR_CLASSES and val_tags are all already in memory.

# FOLDER_ID = "1wHgqAzrEckiyO4wlrSPLZ-hcXGjzKRtd"
# RAW_DIR = "cmz_raw"
# IMG_SIZE = (224, 224)
# BEHAVIOR_CLASSES = ["Eating", "Locomotion", "Resting"]
# if not os.path.exists(RAW_DIR):
#     !gdown --folder "https://drive.google.com/drive/folders/{FOLDER_ID}" -O {RAW_DIR} --quiet
# model = tf.keras.models.load_model("cmz_behavior_classifier.keras")
# model_3class = tf.keras.models.load_model("cmz_behavior_classifier_3class.keras")
# val_tags = ["Eating_25690518092326646", "Locomotion_25690518091210873",
#             "Locomotion_25690518091847591", "Resting_25690519194109670"]


In [ ]:
import cv2


def draw_label(frame, behavior, behavior_conf, state, position="top-left"):
    h, w = frame.shape[:2]
    line1 = f"{behavior} ({behavior_conf*100:.0f}%)"
    line2 = state
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = max(0.6, w / 900)
    thickness = max(1, int(scale * 2))
    (tw1, th1), _ = cv2.getTextSize(line1, font, scale, thickness)
    (tw2, th2), _ = cv2.getTextSize(line2, font, scale * 0.7, max(1, thickness - 1))
    pad = 10
    box_w = max(tw1, tw2) + pad * 2
    box_h = th1 + th2 + pad * 3
    x0 = pad if position == "top-left" else w - box_w - pad
    y0 = pad
    cv2.rectangle(frame, (x0, y0), (x0 + box_w, y0 + box_h), (0, 0, 0), -1)
    cv2.putText(frame, line1, (x0 + pad, y0 + pad + th1), font, scale,
                (255, 255, 255), thickness, cv2.LINE_AA)
    cv2.putText(frame, line2, (x0 + pad, y0 + pad * 2 + th1 + th2), font, scale * 0.7,
                (210, 210, 210), max(1, thickness - 1), cv2.LINE_AA)
    return frame


def annotate_video(video_path, out_path, label_position="top-left"):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    log_rows = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = tf.image.resize(rgb, IMG_SIZE)
        batch = tf.expand_dims(resized, 0)

        inactive_prob = float(model.predict(batch, verbose=0)[0][0])
        state = "Inactive" if inactive_prob >= 0.5 else "Active"

        behavior_probs = model_3class.predict(batch, verbose=0)[0]
        behavior_idx = int(np.argmax(behavior_probs))
        behavior = BEHAVIOR_CLASSES[behavior_idx]
        behavior_conf = float(behavior_probs[behavior_idx])

        frame = draw_label(frame, behavior, behavior_conf, state, label_position)
        writer.write(frame)

        log_rows.append({
            "video": os.path.basename(video_path),
            "frame": frame_idx,
            "time_sec": round(frame_idx / fps, 2),
            "predicted_behavior": behavior,
            "behavior_confidence": round(behavior_conf, 4),
            "active_inactive": state,
        })
        frame_idx += 1

    cap.release()
    writer.release()
    return pd.DataFrame(log_rows)


# ---- run on the 4 held-out validation clips shown in the results above ----
# video_tag format is "<source folder>_<original filename>" (Section 3), so the
# original video file is at cmz_raw/<folder>/<original filename>.mp4.
OUT_DIR = "annotated_videos"
os.makedirs(OUT_DIR, exist_ok=True)
LABEL_POSITION = "top-left"  # change to "top-right" if you prefer

all_logs = []
for tag in sorted(val_tags):
    folder, original_name = tag.split("_", 1)
    src = os.path.join(RAW_DIR, folder, f"{original_name}.mp4")
    out = os.path.join(OUT_DIR, f"{tag}_annotated.mp4")
    df_log = annotate_video(src, out, LABEL_POSITION)
    df_log.to_csv(os.path.join(OUT_DIR, f"{tag}_frame_log.csv"), index=False)
    all_logs.append(df_log)
    print(f"done: {out}  ({len(df_log)} frames)")

video_frame_log = pd.concat(all_logs, ignore_index=True)
video_frame_log.to_csv(os.path.join(OUT_DIR, "all_clips_frame_log.csv"), index=False)
video_frame_log.head(20)


Runs the full two-model inference on every frame of 4 short clips — expect roughly a minute or two total on a T4 GPU (longer on CPU). Download everything as one zip:


In [ ]:
!zip -rq annotated_videos.zip annotated_videos
from google.colab import files
files.download("annotated_videos.zip")


## 13. Pushing this to GitHub

Suggested repo layout:

```
cmz-phase2-behavior-classification/
├── README.md
├── CMZ_Phase2_Behavior_Classification.ipynb   (this notebook)
├── results_table.csv                           (generated above — required deliverable)
├── video_level_table.csv                        (generated above — required deliverable)
├── cmz_behavior_classifier.keras                (generated above — optional, can be large)
├── results_table_3class.csv                     (bonus — Section 11)
├── video_level_table_3class.csv                  (bonus — Section 11)
├── video_level_combined.csv                      (bonus — feeds the dashboard)
├── cmz_behavior_classifier_3class.keras          (bonus — optional, can be large)
├── annotated_videos/                             (Section 12 — behavior overlaid on video)
│   ├── <clip>_annotated.mp4   x4
│   ├── <clip>_frame_log.csv   x4
│   └── all_clips_frame_log.csv
└── figures/
    ├── confusion_matrix.png
    └── training_curves.png
```

From Colab, the fastest path:

1. `File > Download > Download .ipynb` to get this notebook onto your computer (or `File > Save a copy in GitHub` directly from the Colab menu, if you sign in with GitHub).
2. Also download `results_table.csv` and `cmz_behavior_classifier.keras` from the Colab file browser (left sidebar → folder icon → right-click → Download). The `annotated_videos.zip` from Section 12 downloads straight to your computer's Downloads folder when that cell runs — unzip it into `annotated_videos/`.
3. On your computer:
   ```bash
   git init cmz-phase2-behavior-classification
   cd cmz-phase2-behavior-classification
   # copy in the downloaded files
   git add .
   git commit -m "Phase 2: CNN-based active/inactive behavior classification"
   git branch -M main
   git remote add origin <your-empty-github-repo-url>
   git push -u origin main
   ```
4. Paste the resulting GitHub URL into your submission.
